# FingerBall Stage 2F — 3D Hand Skeleton Viewer（21点骨格・手首移動対応版）

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atstuyuki/FingerBall-Analysis/blob/main/notebooks/FingerBall_v2_Stage2F_3D_Hand_Skeleton_Viewer.ipynb)

Stage 2D/2Eの`hand_21_landmarks_long.csv`から、MediaPipe **21点・20本の骨格線をそのままの接続関係で**表示します。点はすべて同じ小サイズ。指先だけを強調しません。

- 画像左上原点の**固定XY座標**で、手首の移動と軌跡が表示されます。
- 時間スライダー、マウス回転、HTMLのYaw/Pitch回転バー、360°回転MP4・PNG・共有ZIPを生成。
- **XYは画像から得た位置、Zは手首を基準とする単眼モデルの相対深度です。** 3Dワールド座標や校正済みカメラ3Dではありません。
- MediaPipe推定結果の正確性は未検証であり、指の実測3D再建ではありません。

In [ ]:
%pip -q install "plotly>=5.18" numpy pandas matplotlib opencv-python-headless tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json,zipfile,math
import numpy as np,pandas as pd,plotly.graph_objects as go
import plotly.io as pio
import matplotlib.pyplot as plt,cv2
from tqdm.auto import tqdm

## 1. Google DriveのCSVを選択・設定

In [ ]:
PROJECT_PATH='/content/drive/MyDrive/母指回内トラッキング'
SOURCE_STAGE='stage2d' # 'stage2d' または 'stage2e'
CSV_PATH='' # 空なら指定Stageから自動選択
VIDEO_STEM='7A5812501DC5F7120DF842D06EE1B74C695459F1'
START_FRAME,END_FRAME=175,210
REPRESENTATIVE_FRAME=198
Z_DEPTH_MULTIPLIER=1.0
COORDINATE_MODE='image'  # 'image': 画像左上原点（推奨）、'wrist': 手首原点
IMAGE_HEIGHT_PX=648  # 動画の高さ。y_normがCSVにあれば自動推定
SHOW_WRIST_TRAJECTORY=True
ROTATION_FRAMES=72
ROTATION_FPS=24
EXPORT_MP4=True
project=Path(PROJECT_PATH)
if CSV_PATH: csv_path=Path(CSV_PATH)
else:
    root=project/f'FingerBall_v2_{SOURCE_STAGE}_results'
    csv_path=root/VIDEO_STEM/'hand_21_landmarks_long.csv'
    if not csv_path.exists():
        choices=sorted(root.glob('*/hand_21_landmarks_long.csv'),key=lambda p:p.stat().st_mtime,reverse=True)
        if not choices:raise FileNotFoundError('CSVが見つかりません。CSV_PATHを指定してください')
        csv_path=choices[0]
if not csv_path.is_file():raise FileNotFoundError(csv_path)
out_dir=project/'FingerBall_v2_stage2f_results'/csv_path.parent.name
out_dir.mkdir(parents=True,exist_ok=True)
print('CSV:',csv_path,'\nOutput:',out_dir)

## 2. 画像原点での座標と手首位置

画像原点モードでは **X=x/W、Y=y/H、Z=−(z_relative−z_wrist)×係数**。手首をXYの原点に戻さないため、時間ごとの手首の画像内移動が表示されます。表示軸は全フレームで固定です。

`COORDINATE_MODE='wrist'` にすると従来の手首原点表示へ切替可能です。いずれもZは相対量であり、実空間のカメラ位置からの距離を示しません。

In [ ]:
df=pd.read_csv(csv_path)
required={'frame','landmark_id','x_px','y_px','z_relative','detected'}
if required-set(df.columns):
    raise ValueError('CSV必須列がありません: '+str(required-set(df.columns)))
if df.duplicated(['frame','landmark_id']).any():
    raise ValueError('frame/landmark_idの重複があります')
for name in ('x_px','y_px','z_relative'):
    df[name]=pd.to_numeric(df[name],errors='coerce')
WIDTH=1206.0
HEIGHT=float(IMAGE_HEIGHT_PX)
if 'x_norm' in df.columns:
    values=df[['x_px','x_norm']].dropna()
    values=values[values.x_norm.abs()>1e-7]
    if len(values):
        WIDTH=float(np.median(values.x_px/values.x_norm))
if 'y_norm' in df.columns:
    values=df[['y_px','y_norm']].dropna()
    values=values[values.y_norm.abs()>1e-7]
    if len(values):
        HEIGHT=float(np.median(values.y_px/values.y_norm))
if not all(np.isfinite(x) and x>0 for x in (WIDTH,HEIGHT)):
    raise ValueError('画像幅と高さが不正です')
if COORDINATE_MODE not in ('image','wrist'):
    raise ValueError("COORDINATE_MODEは'image'か'wrist'を指定")
FINGERS={
 'Thumb':[(0,1),(1,2),(2,3),(3,4)],
 'Index':[(0,5),(5,6),(6,7),(7,8)],
 'Middle':[(0,9),(9,10),(10,11),(11,12)],
 'Ring':[(0,13),(13,14),(14,15),(15,16)],
 'Little':[(0,17),(17,18),(18,19),(19,20)]}
BONE_COLOR='#e5e7eb'
JOINT_COLOR='#22c55e'
WRIST_TRAIL_COLOR='#fb7185'
frames=list(range(START_FRAME,END_FRAME+1))
coords={}
for f in frames:
    g=df[df.frame.eq(f)&df.landmark_id.between(0,20)].sort_values('landmark_id')
    if len(g)!=21 or g.landmark_id.tolist()!=list(range(21)):
        coords[f]=None
        continue
    xyz=g[['x_px','y_px','z_relative']].to_numpy(float)
    if not np.isfinite(xyz).all() or not np.all(g.detected.astype(int).to_numpy()==1):
        coords[f]=None
        continue
    if COORDINATE_MODE=='image':
        x=xyz[:,0]/WIDTH
        y=xyz[:,1]/HEIGHT
    else:
        x=(xyz[:,0]-xyz[0,0])/WIDTH
        y=-(xyz[:,1]-xyz[0,1])/HEIGHT
    z=-(xyz[:,2]-xyz[0,2])*Z_DEPTH_MULTIPLIER
    coords[f]=np.column_stack([x,y,z])
valid=[f for f in frames if coords[f] is not None]
if not valid:
    raise RuntimeError('有効な21点フレームがありません')
if REPRESENTATIVE_FRAME not in valid:
    old=REPRESENTATIVE_FRAME
    REPRESENTATIVE_FRAME=min(valid,key=lambda f:abs(f-old))
    print('代表フレーム変更:',old,'→',REPRESENTATIVE_FRAME)
Z_MAX=max(.05,max(float(np.max(np.abs(coords[f][:,2]))) for f in valid)*1.3)
if COORDINATE_MODE=='image':
    X_RANGE=[-.03,1.03]
    Y_RANGE=[-.03,1.03]
else:
    xy_max=max(.1,max(float(np.max(np.abs(coords[f][:,:2]))) for f in valid)*1.2)
    X_RANGE=[-xy_max,xy_max]
    Y_RANGE=[-xy_max,xy_max]
Z_RANGE=[-Z_MAX,Z_MAX]
ASPECT=dict(x=1,y=HEIGHT/WIDTH,z=.60)
print('valid:',len(valid),'/',len(frames),'image:',int(round(WIDTH)),int(round(HEIGHT)),
      'coordinate mode:',COORDINATE_MODE)
print('wrist positions (first/last valid):',
      np.round(coords[valid[0]][0,:2],4),np.round(coords[valid[-1]][0,:2],4))
print('wrist XY displacement:',
      np.round(coords[valid[-1]][0,:2]-coords[valid[0]][0,:2],4))

## 3. 21点MediaPipe骨格：時間スライダー・画像内手首移動・自由回転

骨格線は指ごとの20本の接続線、関節は**同一サイズの緑点21個**で表示します。赤い細線はフレーム経過に伴う手首の画像内軌跡（参考表示）です。画像枠に対して手の位置が変化することを確認してください。

In [ ]:
def mediapipe_traces(frame_id):
    p=coords.get(frame_id)
    result=[]
    # 各frameでtraceの本数・順番・種類を固定（Plotly slider互換）
    for finger,edges in FINGERS.items():
        x=[];y=[];z=[]
        for a,b in edges:
            if p is None:
                x.extend([None,None,None]);y.extend([None,None,None]);z.extend([None,None,None])
            else:
                x.extend([float(p[a,0]),float(p[b,0]),None])
                y.extend([float(p[a,1]),float(p[b,1]),None])
                z.extend([float(p[a,2]),float(p[b,2]),None])
        result.append(go.Scatter3d(x=x,y=y,z=z,mode='lines',
            line=dict(color=BONE_COLOR,width=6),name=finger,showlegend=False))
    # 大きな指先マーカーを廃止。21点はすべて同一の小さなマーカー。
    result.append(go.Scatter3d(
        x=[] if p is None else p[:,0],
        y=[] if p is None else p[:,1],
        z=[] if p is None else p[:,2],
        mode='markers',marker=dict(size=3,color=JOINT_COLOR),
        name='21 joints',showlegend=True))
    # 手首を他の点と同じ大きさで赤く表示
    result.append(go.Scatter3d(
        x=[] if p is None else [float(p[0,0])],
        y=[] if p is None else [float(p[0,1])],
        z=[] if p is None else [float(p[0,2])],
        mode='markers',marker=dict(size=3,color='#ef4444'),
        name='wrist',showlegend=True))
    trail=[]
    if COORDINATE_MODE=='image' and SHOW_WRIST_TRAJECTORY:
        trail=[coords[k][0] for k in frames if k<=frame_id and coords[k] is not None]
    result.append(go.Scatter3d(
        x=[float(q[0]) for q in trail],y=[float(q[1]) for q in trail],
        z=[float(q[2]) for q in trail],mode='lines',
        line=dict(color=WRIST_TRAIL_COLOR,width=3),
        name='wrist path',showlegend=True))
    if COORDINATE_MODE=='image':
        bx=[0,1,1,0,0];by=[0,0,1,1,0]
    else:
        bx=[];by=[]
    result.append(go.Scatter3d(x=bx,y=by,z=[0]*len(bx),
        mode='lines',line=dict(color='#64748b',width=2),
        name='image boundary',showlegend=True))
    return result

fig=go.Figure(
    data=mediapipe_traces(START_FRAME),
    frames=[go.Frame(name=str(f),data=mediapipe_traces(f),
              layout=go.Layout(title=f'Frame {f} | '+('21 landmarks' if coords[f] is not None else 'No detection')))
            for f in frames])
fig.update_layout(
    template='plotly_dark',height=750,
    title='FingerBall — MediaPipe 21-point skeleton | image-origin XY + wrist-relative Z',
    scene=dict(
       xaxis=dict(title='Image X (left to right)',range=X_RANGE),
       yaxis=dict(title='Image Y (top to bottom)',range=Y_RANGE),
       zaxis=dict(title='Relative Z from wrist',range=Z_RANGE),
       aspectmode='manual',aspectratio=ASPECT,
       camera=dict(eye=dict(x=1.8,y=1.5,z=1.2)),dragmode='orbit'),
    uirevision='preserve_camera',
    sliders=[dict(currentvalue=dict(prefix='Frame '),steps=[
        dict(label=str(f),method='animate',args=[[str(f)],
             dict(mode='immediate',frame=dict(duration=0,redraw=True),
                  transition=dict(duration=0))]) for f in frames])],
    updatemenus=[dict(type='buttons',showactive=False,buttons=[
        dict(label='Play',method='animate',
             args=[None,dict(frame=dict(duration=250,redraw=True),fromcurrent=True)]),
        dict(label='Pause',method='animate',
             args=[[None],dict(mode='immediate',frame=dict(duration=0,redraw=True))])])])
fig.show(config=dict(scrollZoom=True,displaylogo=False))

## 4. 画像座標原点を維持したインタラクティブHTMLを保存

HTML上の時間スライダーで手首の画像内移動を確認し、Yaw/Pitchで360°回転できます。画像枠はZ=0の平面上に表示します。

In [ ]:
html_path=out_dir/'FingerBall_Stage2F_interactive_3D.html'
html=pio.to_html(fig,include_plotlyjs=True,full_html=True,div_id='fingerball3d')
controls="""<div style="padding:16px;background:#111827;color:white;font-family:sans-serif">
横回転 Yaw <input id="fbYaw" type="range" min="-180" max="180" value="40"><span id="yawVal">40°</span>
縦回転 Pitch <input id="fbPitch" type="range" min="-180" max="180" value="25"><span id="pitchVal">25°</span>
<button id="fbFront">正面</button><button id="fbSide">側面</button><button id="fbTop">上面</button>
<div style="font-size:12px">画像左上原点・XYは画像位置、Zは手首相対。実空間のカメラ座標系ではありません。</div></div>"""
script="""<script>
(function(){
const gd=document.getElementById('fingerball3d'),
y=document.getElementById('fbYaw'),p=document.getElementById('fbPitch');
function redraw(){
const a=Number(y.value)*Math.PI/180,b=Number(p.value)*Math.PI/180,r=2.4;
Plotly.relayout(gd,{'scene.camera.eye':{x:r*Math.cos(b)*Math.cos(a),y:r*Math.cos(b)*Math.sin(a),z:r*Math.sin(b)}});
document.getElementById('yawVal').textContent=y.value+'°';
document.getElementById('pitchVal').textContent=p.value+'°';
}
y.addEventListener('input',redraw);p.addEventListener('input',redraw);
function set(a,b){y.value=a;p.value=b;redraw();}
document.getElementById('fbFront').onclick=()=>set(0,0);
document.getElementById('fbSide').onclick=()=>set(90,0);
document.getElementById('fbTop').onclick=()=>set(0,89);
})();
</script>"""
html=html.replace('<body>','<body>'+controls,1).replace('</body>',script+'</body>',1)
html_path.write_text(html,encoding='utf-8')
print('HTML:',html_path,round(html_path.stat().st_size/1e6,2),'MB')

## 5. 360°回転MP4・0/90/180/270°PNG

Plotlyビューアと同様に、白い20本の骨格線と同一サイズの21点のみを描画します。画像原点モードでは画像枠も表示します。MP4は代表フレームを固定し、**表示視点だけ**を回転させます。

In [ ]:
def render_view(p,az,elev):
    f=plt.figure(figsize=(5.7,5.7),dpi=95,facecolor='#101827')
    ax=f.add_subplot(111,projection='3d')
    ax.set_facecolor('#101827')
    for finger,edges in FINGERS.items():
        for a,b in edges:
            v=p[[a,b]]
            ax.plot(v[:,0],v[:,1],v[:,2],color='white',linewidth=2.6)
    ax.scatter(p[:,0],p[:,1],p[:,2],s=18,color=JOINT_COLOR,depthshade=False)
    if COORDINATE_MODE=='image':
        ax.plot([0,1,1,0,0],[0,0,1,1,0],[0]*5,
                color='#64748b',linewidth=1.2)
    ax.set(xlim=X_RANGE,ylim=Y_RANGE,zlim=Z_RANGE,
           xlabel='Image X',ylabel='Image Y',zlabel='wrist-relative Z')
    ax.set_box_aspect((1,HEIGHT/WIDTH,.60))
    ax.view_init(elev=elev,azim=az)
    ax.tick_params(labelsize=7,colors='white')
    for label in (ax.xaxis.label,ax.yaxis.label,ax.zaxis.label):
        label.set_color('white')
    f.suptitle(f'Frame {REPRESENTATIVE_FRAME} | MediaPipe 21-point skeleton',color='white')
    f.canvas.draw()
    rgba=np.asarray(f.canvas.buffer_rgba())
    bgr=cv2.cvtColor(rgba[:,:,:3].copy(),cv2.COLOR_RGB2BGR)
    plt.close(f)
    return bgr

reference=coords[REPRESENTATIVE_FRAME]
preview=render_view(reference,30,15)
hh,ww=preview.shape[:2]
def make_rotation(path,kind):
    writer=cv2.VideoWriter(str(path),cv2.VideoWriter_fourcc(*'mp4v'),ROTATION_FPS,(ww,hh))
    if not writer.isOpened():
        raise RuntimeError('MP4 writer failed')
    try:
        for i in tqdm(range(ROTATION_FRAMES),desc=kind):
            degrees=i*360/ROTATION_FRAMES
            az,elev=(degrees,15) if kind=='yaw' else (30,degrees)
            bgr=render_view(reference,az,elev)
            cv2.putText(bgr,f'{kind.upper()} {degrees:.0f} deg',(16,hh-18),
                cv2.FONT_HERSHEY_SIMPLEX,.65,(255,255,255),2)
            writer.write(bgr)
    finally:
        writer.release()
yaw_mp4=out_dir/f'frame_{REPRESENTATIVE_FRAME}_yaw_360.mp4'
pitch_mp4=out_dir/f'frame_{REPRESENTATIVE_FRAME}_pitch_360.mp4'
if EXPORT_MP4:
    make_rotation(yaw_mp4,'yaw')
    make_rotation(pitch_mp4,'pitch')
png_paths=[]
for kind in ['yaw','pitch']:
    for deg in [0,90,180,270]:
        az,el=(deg,15) if kind=='yaw' else (30,deg)
        p=out_dir/f'frame_{REPRESENTATIVE_FRAME}_{kind}_{deg:03d}.png'
        cv2.imwrite(str(p),render_view(reference,az,el))
        png_paths.append(p)
print('MP4:',yaw_mp4,pitch_mp4)

## 6. 共有用ZIPと解析条件を保存

In [ ]:
report={'source_csv':str(csv_path),'frames':[START_FRAME,END_FRAME],
        'valid_frames':valid,'representative_frame':REPRESENTATIVE_FRAME,
        'image_width_px':WIDTH,'image_height_px':HEIGHT,
        'coordinate_mode':COORDINATE_MODE,
        'visual_style':'MediaPipe 21 uniform small joints + white skeleton bones',
        'XY_origin':'image top-left (0,0)' if COORDINATE_MODE=='image' else 'wrist',
        'Z_origin':'wrist-relative; absolute camera depth unavailable','relative_depth_multiplier':Z_DEPTH_MULTIPLIER,
        'limitations':['Single-view pseudo-3D, not metric 3D',
         'Depth is MediaPipe relative Z; no ground truth for unseen surface',
         'No validated ball release or joint angles',
         'Missing frames not imputed',
         'Wrist image motion is relative to the camera frame, not metric world translation',
         'Image origin is not calibrated camera 3D space']}
report_file=out_dir/'stage2f_report.json'
report_file.write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
readme=out_dir/'README_for_trainer.txt'
readme.write_text('FingerBall Stage 2F: 手指骨格疑似3Dビューア\n'
  'HTMLの時間スライダーとYaw/Pitchスライダーで操作できます。\n'
  'MP4は代表フレームの横360度／縦360度回転表示です。\n'
  '奥行きは単眼MediaPipeの相対Zであり、実空間で計測した3Dではありません。\n'
  '共有前に個人情報・研究倫理・データ提供者の許諾をご確認ください。\n',encoding='utf-8')
package=out_dir/'FingerBall_Stage2F_3D_hand_viewer_package.zip'
assets=[html_path,report_file,readme]+png_paths
if EXPORT_MP4:assets += [yaw_mp4,pitch_mp4]
with zipfile.ZipFile(package,'w',zipfile.ZIP_DEFLATED) as z:
    for item in assets:
        if not item.exists():raise FileNotFoundError(item)
        z.write(item,arcname=item.name)
print('ZIP:',package,'MB:',round(package.stat().st_size/1e6,2))
print('Results:',out_dir)

## 利用手順

1. Google Driveへ接続してStage 2D/2Eの21点CSVを読み込み。
2. Notebookの3D表示でフレームを選び、マウスで360度回転。
3. 出力HTMLを開けば **時間スライダー＋Yaw/Pitch回転バー**が使えます。
4. 代表フレームのYaw/Pitch回転MP4とZIPを確認してから共有。